# R3 - Experiments requested in the supervisor's comments (Comment_ver_2)

Attach the dataset **`snapshot-fd-code-data`** (the same one used by N1-N6 and S).
Settings: Accelerator **None (CPU)**. Parts F and G need OpenDSS: attach the dataset **`opendss-wheels`** (works without Internet) or turn Internet ON.

| Part | Comment | What is run |
|---|---|---|
| A | C1 | Feature ablation f1-f4 at the 1% TVE class (old numbers came from the +-2% setting) |
| B | C6, C11 | Table 3 rerun under ONE protocol (2 seeds x 5 folds) + Cover-Hart estimate per column |
| C | C6 | DMP k=3, FPS, LightGBM: one number for Table 5 and Table 7 |
| D | C3 | N-1 two ways: lost sensor frozen (no retraining) vs retrained on the remaining sensors |
| E | C11 | Cover-Hart estimate against the number of samples (shows the finite-sample bias) |
| F | C7 | New simulations with R_f up to 1000 ohm, accuracy per R_f bin |
| G | C8 | New simulations with inverter-based DER on the 34- and 123-node feeders |
| I | C6 | Expected calibration error (ECE) with the DMP sensors |
| H | C8 | Grid-code inverter fault model (iterative current source): current limit, k-factor reactive current, + negative sequence, momentary cessation |

Every part saves its result to `/kaggle/working/results/R3_*.json` and the last cell zips them.
Parts are independent: set the flags in the first cell to run only some of them.

In [ ]:
# ============================== SETUP ==============================
SMOKE = False            # True = small quick test (minutes)
RUN = dict(A=False, B=False, C=False, D=False, E=False, F=False, G=False, H=True, I=True)  # A-G already done
H_STRATEGIES = ("limit", "kfactor", "kfactor_neg", "mc")  # part H inverter fault strategies
H_PEN = 0.6              # part H: inverter output, share of feeder load
H_IMAX = 1.2             # part H: inverter current limit (pu of rated current)
N_SIM = 12000            # samples per new simulation (parts F, G); same as the original data sets
RF_MAX_NEW = 1000.0      # part F: upper fault resistance (ohm)
DER_LEVELS = (0.3, 0.6)  # part G: inverter DER penetration (share of feeder load)

import os, sys, glob, json, time, shutil, warnings
warnings.filterwarnings("ignore")
t0 = time.time()
_c = sorted(glob.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
assert _c, "Attach the dataset 'snapshot-fd-code-data'"
DATA = os.path.dirname(_c[0])
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results"); os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS; os.environ["K4_DATA"] = DATA
sys.path.insert(0, DATA)

if RUN["F"] or RUN["G"] or RUN.get("H"):
    try:
        import opendssdirect  # noqa
    except ImportError:
        whl = sorted(glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True))
        if whl:   # offline: dataset 'opendss-wheels'
            os.system("pip install -q --no-index --no-deps " + " ".join(f'"{w}"' for w in whl))
        else:     # online: needs Internet ON
            os.system("pip install -q OpenDSSDirect.py")
try:
    import opendssdirect  # noqa
    HAVE_DSS = True
except ImportError:
    import types; sys.modules["opendssdirect"] = types.ModuleType("opendssdirect"); HAVE_DSS = False

import numpy as np, pandas as pd
import k4lib as K, features as F, models as M, placement2 as Q
from sklearn.base import clone
print("DATA", DATA, "| OpenDSS", HAVE_DSS, "| 1% TVE noise", K.NOISE["1% TVE"])

SEEDS = (0,) if SMOKE else (0, 1)
FEEDERS = [13] if SMOKE else [13, 34, 123]
NZ = "1% TVE"

def save(name, obj):
    with open(os.path.join(RESULTS, name + ".json"), "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    print("saved", name)

def small(ds, n=1500):
    # SMOKE: keep a stratified subset
    if not SMOKE: return ds
    idx = np.random.RandomState(0).permutation(ds.n)[:n]
    for a in ("Vpre", "Vf", "Ipre", "If", "ftype", "fbus", "rf", "y"):
        setattr(ds, a, getattr(ds, a)[idx])
    ds.n = len(idx); return ds

def dmp_sets(nb, k=3):
    # the sensor sets used in the manuscript (Table 7), recorded by e14
    b = K.baselines(nb, k)
    if "DMP (proposed)" in b:
        return {"DMP": b["DMP (proposed)"]["buses"], "DMP N-1": b["DMP N-1 (proposed)"]["buses"]}
    raise RuntimeError("e14_placement_baselines.json not found in the dataset")

PAPER_3BUS = ["650", "675", "684"]

## A. Feature ablation at the 1% TVE class (comment C1)
13-node feeder, the reference three-bus set and all 14 buses, AGG features. Each of f1-f4 is removed in
turn. RF and SVM (the two models quoted in the text) plus LightGBM and XGBoost. Mean +- SD over 2 seeds x 5 folds.

In [ ]:
if RUN["A"]:
    ds = small(K.dataset(13)); rows = []
    for sensing, obs in [("3bus", ds.bus_ids(PAPER_3BUS)), ("all", list(range(len(ds.buses))))]:
        X = K.features_of(ds, obs, 0, NZ, "AGG")
        for name in ("RF", "SVM", "LGBM", "XGB"):
            for drop in [None, 0, 1, 2, 3]:
                Xd = X if drop is None else np.delete(X, drop, 1)
                r = K.folds_acc(Xd, ds.y, ds.rf, seeds=SEEDS, model=M.zoo()[name])
                rows.append(dict(sensing=sensing, model=name, removed="none" if drop is None else f"f{drop+1}",
                                 acc=r["acc"], sd=r["acc_sd"]))
                print(sensing, name, rows[-1]["removed"], f"{100*r['acc']:.1f} +- {100*r['acc_sd']:.1f}", flush=True)
    A = pd.DataFrame(rows)
    print(A.pivot_table(index=["sensing", "model"], columns="removed", values="acc").mul(100).round(1))
    save("R3_A_ablation", rows)

## B. Table 3 under one protocol + Cover-Hart estimate (comments C6, C11)
Seven classifiers x four feature sets, 13-node feeder, all 14 buses and the three-bus set, 1% TVE,
**2 seeds x 5 folds** for every cell. The SD range of the tree ensembles and the 3-bus FPS value
(text says 86.3/88.2) are printed at the end.

In [ ]:
if RUN["B"]:
    ds = small(K.dataset(13)); rows = []
    names = ["RF", "XGB", "LGBM", "MLP", "SVM", "kNN", "LR"]
    for sensing, obs in [("all", list(range(len(ds.buses)))), ("3bus", ds.bus_ids(PAPER_3BUS))]:
        for rep in ("AGG", "PPA", "SEQ", "FPS"):
            X = K.features_of(ds, obs, 0, NZ, rep)
            for name in names:
                r = K.folds_acc(X, ds.y, ds.rf, seeds=SEEDS, model=M.zoo()[name])
                rows.append(dict(sensing=sensing, rep=rep, model=name, acc=r["acc"], sd=r["acc_sd"], hif=r["hif"]))
                print(sensing, rep, name, f"{100*r['acc']:.1f} +- {100*r['acc_sd']:.1f}", flush=True)
            e1, lb = F.bayes_lower_bound(X, ds.y, 3)
            rows.append(dict(sensing=sensing, rep=rep, model="Cover-Hart", acc=1 - lb, nn1_err=e1))
    B = pd.DataFrame(rows)
    print(B.pivot_table(index="model", columns=["sensing", "rep"], values="acc").mul(100).round(1).to_string())
    tree = B[B.model.isin(["RF", "XGB", "LGBM"])]
    print("SD range of tree ensembles: %.1f - %.1f points" % (100 * tree.sd.min(), 100 * tree.sd.max()))
    save("R3_B_table3", rows)

## C. One number for DMP, k = 3, FPS, LightGBM, 1% TVE (comment C6: Table 5 vs Table 7)
Same sensors, same model (FAST LightGBM, as in the placement study), same folds. Also micro-PMU,
which settles the "96-99%" sentence of the abstract.

In [ ]:
if RUN["C"]:
    rows = []
    for nb in FEEDERS:
        ds = small(K.dataset(nb)); sets = dmp_sets(nb)
        for meth, buses in sets.items():
            obs = K.rows_for_buses(ds, buses)
            for noise in ("1% TVE", "uPMU"):
                r = K.folds_acc(K.features_of(ds, obs, 0, noise, "FPS"), ds.y, ds.rf, seeds=SEEDS)
                rows.append(dict(feeder=nb, method=meth, buses=buses, noise=noise, acc=r["acc"], sd=r["acc_sd"], hif=r["hif"]))
                print(nb, meth, buses, noise, f"{100*r['acc']:.1f} +- {100*r['acc_sd']:.1f}  HIF {100*r['hif']:.1f}", flush=True)
    save("R3_C_dmp_reference", rows)

## D. Loss of one sensor, two protocols (comment C3)
* **frozen**: the model trained with 3 sensors is kept; the lost sensor reports its pre-fault value
  (this is the protocol of Table 11 / Fig. 6).
* **retrained**: a new model is trained on the two remaining sensors (this is what "at most 0.6 points" measured).
Mean and worst case over the three possible losses, for DMP and DMP N-1.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
if RUN["D"]:
    rows = []
    for nb in FEEDERS:
        ds = small(K.dataset(nb))
        for meth, buses in dmp_sets(nb).items():
            obs = K.rows_for_buses(ds, buses)
            Vp, Vf = K.measure(ds, obs, 0, NZ)
            X = F.fps(Vp, Vf)
            nom, frozen = [], {j: [] for j in range(3)}
            for s in SEEDS:
                for tr, te in StratifiedKFold(5, shuffle=True, random_state=s).split(X, ds.y):
                    m = LGBMClassifier(**K.FAST).fit(X[tr], ds.y[tr])
                    nom.append((m.predict(X[te]) == ds.y[te]).mean())
                    for j in range(3):
                        Vl = Vf[te].copy(); Vl[:, j] = Vp[te][:, j]
                        frozen[j].append((m.predict(F.fps(Vp[te], Vl)) == ds.y[te]).mean())
            retr = {}
            for j in range(3):
                keep = [o for i, o in enumerate(obs) if i != j]
                retr[j] = K.folds_acc(K.features_of(ds, keep, 0, NZ, "FPS"), ds.y, seeds=SEEDS)["acc"]
            fz = {j: float(np.mean(v)) for j, v in frozen.items()}
            row = dict(feeder=nb, method=meth, buses=buses, nominal=float(np.mean(nom)),
                       frozen_mean=float(np.mean(list(fz.values()))), frozen_worst=min(fz.values()),
                       retrained_mean=float(np.mean(list(retr.values()))), retrained_worst=min(retr.values()),
                       frozen_each=fz, retrained_each=retr)
            rows.append(row)
            print(nb, meth, {k: round(100 * v, 1) for k, v in row.items() if isinstance(v, float)}, flush=True)
    save("R3_D_n1_protocols", rows)

## E. Cover-Hart estimate against sample size (comment C11)
The 1-NN leave-one-out error is computed on random subsets of growing size. If the implied accuracy
ceiling still moves with N, the estimate is not converged and must be called an asymptotic estimate.

In [ ]:
if RUN["E"]:
    ds = K.dataset(13); rows = []
    sizes = [500, 1000, 2000] if SMOKE else [500, 1000, 2000, 4000, 6000, ds.n]
    for sensing, obs in [("all", list(range(len(ds.buses)))), ("3bus", ds.bus_ids(PAPER_3BUS))]:
        for rep in ("AGG", "PPA", "SEQ", "FPS"):
            X = K.features_of(ds, obs, 0, NZ, rep)
            for n in sizes:
                vals = []
                for r in range(3 if n < ds.n else 1):
                    idx = np.random.RandomState(r).permutation(ds.n)[:n]
                    vals.append(1 - F.bayes_lower_bound(X[idx], ds.y[idx], 3)[1])
                rows.append(dict(sensing=sensing, rep=rep, n=int(n), ch_acc=float(np.mean(vals))))
                print(sensing, rep, n, f"C-H ceiling {100*np.mean(vals):.1f}", flush=True)
    save("R3_E_cover_hart_vs_n", rows)

## F. Fault resistance up to 1000 ohm (comment C7)
New OpenDSS simulations (same generator as the paper, only `rf_range` changed). Accuracy per log R_f bin
with the DMP three-sensor set, FPS, LightGBM, at 1% TVE and micro-PMU noise.

In [ ]:
SIMDIR = os.path.join(WORK, "sim_new"); os.makedirs(SIMDIR, exist_ok=True)

def get_simlib():
    # copy the feeder models to a writable place (OpenDSS changes the working directory)
    import importlib, simlib
    fd = os.path.join(WORK, "feeders")
    if not os.path.isdir(fd):
        shutil.copytree(os.path.join(DATA, "data", "feeders"), fd)
        # Linux is case-sensitive: the .dss files redirect to "x.dss" while the file is "x.DSS"
        for d, _, fs in os.walk(fd):
            for f in fs:
                for alt in (f[:-4] + ".dss", f[:-4] + ".DSS") if f.lower().endswith(".dss") else ():
                    if not os.path.exists(os.path.join(d, alt)):
                        shutil.copy(os.path.join(d, f), os.path.join(d, alt))
    simlib.FEEDERS.update({
        13: os.path.join(fd, "IEEETestCases", "13Bus", "IEEE13Nodeckt.dss"),
        34: os.path.join(fd, "IEEETestCases", "34Bus", "ieee34Mod1.dss"),
        123: os.path.join(fd, "IEEETestCases", "123Bus", "IEEE123Master.dss")})
    return simlib

def simulate(nb, tag, **kw):
    path = os.path.join(SIMDIR, f"ieee{nb}_{tag}.npz")
    if not os.path.isfile(path):
        t = time.time()
        n = get_simlib().generate(nb, 600 if SMOKE else N_SIM, 2026, path, **kw)
        print(f"simulated ieee{nb}_{tag}: {n} samples, {(time.time()-t)/60:.1f} min", flush=True)
    return F.Dataset(path)

if RUN["F"] and not HAVE_DSS:
    print("SKIP part F: OpenDSS is not installed. Attach the dataset 'opendss-wheels' or turn Internet ON.")
if RUN["F"] and HAVE_DSS:
    edges = np.array([1e-3, 1e-2, 1e-1, 1, 10, 32, 100, 316, 1000.0001])
    rows = []
    for nb in FEEDERS:
        ds = simulate(nb, f"rf{int(RF_MAX_NEW)}", rf_range=(1e-3, RF_MAX_NEW))
        obs = K.rows_for_buses(ds, dmp_sets(nb)["DMP"])
        b = np.digitize(ds.rf, edges[1:-1])
        for noise in ("1% TVE", "uPMU"):
            r = K.folds_acc(K.features_of(ds, obs, 0, noise, "FPS"), ds.y, ds.rf, seeds=SEEDS, return_oof=True)
            ok = r["oof"] == ds.y
            by = [float(ok[b == i].mean()) if (b == i).any() else None for i in range(len(edges) - 1)]
            rows.append(dict(feeder=nb, noise=noise, acc=r["acc"], sd=r["acc_sd"], bins=edges.tolist(),
                             acc_by_bin=by, n_by_bin=[int((b == i).sum()) for i in range(len(edges) - 1)]))
            print(nb, noise, f"overall {100*r['acc']:.1f}", [None if v is None else round(100 * v, 1) for v in by], flush=True)
    save("R3_F_rf1000", rows)

## G. Inverter-based DER on the IEEE feeders (comment C8)
Three current-limited inverters (OpenDSS Generator model 7, as in the 13-node DER study of the paper),
total output 30% and 60% of feeder load, R_f up to 100 ohm. Two questions: accuracy when the model is
trained and tested with DER, and when it is trained without DER and tested with DER.

In [ ]:
if RUN["G"] and not HAVE_DSS:
    print("SKIP part G: OpenDSS is not installed. Attach the dataset 'opendss-wheels' or turn Internet ON.")
if RUN["G"] and HAVE_DSS:
    rows = []
    for nb in FEEDERS:
        base = small(K.dataset(nb)); buses = dmp_sets(nb)["DMP"]
        for pen in DER_LEVELS:
            ds = simulate(nb, f"der{int(100*pen)}", der_penetration=pen)
            obs, obs0 = K.rows_for_buses(ds, buses), K.rows_for_buses(base, buses)
            for noise in ("1% TVE", "uPMU"):
                for rep in ("SEQ", "FPS"):
                    X = K.features_of(ds, obs, 0, noise, rep)
                    r = K.folds_acc(X, ds.y, ds.rf, seeds=SEEDS)
                    X0 = K.features_of(base, obs0, 0, noise, rep)
                    m = LGBMClassifier(**K.FAST).fit(X0, base.y)
                    ok = m.predict(X) == ds.y
                    rows.append(dict(feeder=nb, der=pen, noise=noise, rep=rep, same_acc=r["acc"], same_sd=r["acc_sd"],
                                     same_hif=r["hif"], transfer_acc=float(ok.mean()),
                                     transfer_hif=float(ok[ds.rf > 10].mean())))
                    print(nb, pen, noise, rep, f"train+test DER {100*r['acc']:.1f} | no-DER -> DER {100*ok.mean():.1f}", flush=True)
    save("R3_G_der_ieee", rows)

## H. Grid-code inverter fault model (comment C8, second step)
Each inverter is a three-phase voltage-dependent current source (three OpenDSS Isource elements), re-solved
iteratively until the terminal voltage changes by less than 0.001 pu (Seuss et al. 2015; Furlaneto et al. 2021;
Haddadi et al. 2024). Four fault strategies: `limit` (constant power, positive sequence, current limit),
`kfactor` (reactive-current priority, k = 3), `kfactor_neg` (+ negative-sequence current leading V2 by 90 deg,
IEEE 2800 style), `mc` (momentary cessation below 0.5 pu). Current limit 1.2 pu. Same three inverter sites as G.
Evaluation: DMP sensors, 1% TVE and micro-PMU, SEQ and FPS. Three training settings: same strategy (5-fold),
trained without DER, trained with the `limit` strategy.

In [ ]:
INVLIB = r"""# Grid-code inverter fault model for the OpenDSS snapshot generator (part H, comment C8).
#
# Each inverter is a three-phase voltage-dependent current source built from three single-phase
# OpenDSS Isource elements. Its phase currents are recomputed from the terminal voltage and the
# network is re-solved until the terminal voltage changes by less than TOL pu, following the
# iterative current-source approach of Seuss et al. (2015, OpenDSS) and the voltage-controlled
# current source used in phasor short-circuit studies (Furlaneto et al. 2021; Haddadi et al. 2024).
#
# Strategies during the fault (positive-sequence voltage |V1| below VRT_ON):
#   "limit"     constant active power, positive sequence only, current limited to IMAX
#   "kfactor"   reactive-current priority: dIq1 = K * (VRT_ON - |V1|) * Ir, rest active (k = 3, Haddadi 2024)
#   "kfactor_neg" as "kfactor" plus negative-sequence current I2 = K * |V2| * Ir leading V2 by 90 deg
#               (IEEE 2800 range 90-100 deg, Javid et al. 2025)
#   "mc"        momentary cessation: no current when |V1| < MC_V (0.5 pu, Omran et al. 2025), else "limit"
# Current limit: IMAX = 1.2 Ir by default (1.1-1.3 pu typical, PSRC 2020); reactive and negative-sequence
# currents have priority and are scaled down together when they alone exceed IMAX.
#
import numpy as np
import opendssdirect as dss
import simlib

A = np.exp(2j * np.pi / 3)
K = 3.0
VRT_ON = 0.9
MC_V = 0.5
TOL = 1e-3
MAX_IT = 40
DAMP = 0.5


def seq(v):
    a, b, c = v
    return (a + b + c) / 3, (a + A * b + A * A * c) / 3, (a + A * A * b + A * c) / 3


def phases(i1, i2):
    return np.array([i1 + i2, A * A * i1 + A * i2, A * i1 + A * A * i2])


class InvFeeder(simlib.Feeder):
    def __init__(self, n_bus, der_penetration, strategy="kfactor", imax=1.2, seed=0):
        super().__init__(n_bus, der_penetration=0.0, seed=seed)
        self.strategy, self.imax = strategy, imax
        # same inverter sites as simlib (three-phase buses, not the substation, seeded)
        rng = np.random.RandomState(seed)
        three_ph = [b for b in self.buses if len(self.bus_phases[b]) == 3 and b != simlib.SUBSTATION[n_bus]]
        self.der_buses = list(rng.choice(three_ph, size=min(3, len(three_ph)), replace=False))
        self.p_kw = der_penetration * self.total_load_kw / len(self.der_buses)
        self.inv = []
        for i, b in enumerate(self.der_buses):
            dss.Circuit.SetActiveBus(b)
            vb = dss.Bus.kVBase() * 1000.0                      # line-to-neutral base (V)
            ir = self.p_kw * 1000.0 / (3 * vb)                    # rated current per phase (A), S = P
            for p in (1, 2, 3):
                dss.Command(f"New Isource.INV{i}_{p} phases=1 bus1={b}.{p} amps=0 angle=0")
            self.inv.append(dict(bus=b, vb=vb, ir=ir, cur=np.zeros(3, complex)))
        self.iters, self.ok = [], []
        self._base_solve()

    # ------------------------------------------------------------------ inverter control
    def _vterm(self, inv):
        dss.Circuit.SetActiveBus(inv["bus"])
        v = np.array(dss.Bus.Voltages()); nodes = dss.Bus.Nodes()
        vv = v[0::2] + 1j * v[1::2]
        return np.array([vv[nodes.index(p)] for p in (1, 2, 3)]) / inv["vb"]

    def _target(self, inv, v, fault):
        v1, v2 = seq(v)[1:]
        m1 = abs(v1); u1 = v1 / m1 if m1 > 1e-6 else 1.0
        imax = self.imax
        p_pu = 1.0                                             # active power in pu of rating
        id_ref = p_pu / max(m1, 0.05)                          # constant power
        iq, i2 = 0.0, 0j
        st = self.strategy if fault else "limit"
        if st == "mc" and m1 < MC_V:
            return np.zeros(3, complex)
        if st in ("kfactor", "kfactor_neg") and m1 < VRT_ON:
            iq = K * (VRT_ON - m1)
            if st == "kfactor_neg":
                i2 = K * abs(v2) * np.exp(1j * (np.angle(v2) + np.pi / 2))
        # priority currents (reactive, negative) limited first
        i1q = -1j * iq * u1                                    # lags V1 by 90 deg: injects reactive power
        pr = phases(i1q, i2)
        peak = np.abs(pr).max()
        if peak > imax:
            s = imax / peak; i1q, i2 = i1q * s, i2 * s; pr = pr * s
        # largest active current that keeps every phase within imax (bisection)
        lo, hi = 0.0, id_ref
        for _ in range(30):
            mid = (lo + hi) / 2
            if np.abs(phases(i1q + mid * u1, i2)).max() <= imax + 1e-9:
                lo = mid
            else:
                hi = mid
        return phases(i1q + lo * u1, i2) * inv["ir"]

    def _set(self, inv, i, cur):
        for p in range(3):
            dss.Command(f"Isource.INV{i}_{p + 1}.amps={abs(cur[p]):.6f} angle={np.degrees(np.angle(cur[p])):.6f}")
        inv["cur"] = cur

    def _iterate(self, fault):
        last = [self._vterm(inv) for inv in self.inv]
        for it in range(1, MAX_IT + 1):
            for i, inv in enumerate(self.inv):
                tgt = self._target(inv, last[i], fault)
                self._set(inv, i, DAMP * tgt + (1 - DAMP) * inv["cur"] if it > 1 else tgt)
            dss.Solution.Solve()
            now = [self._vterm(inv) for inv in self.inv]
            dv = max(np.abs(a - b).max() for a, b in zip(now, last))
            last = now
            if dv < TOL:
                return it, True
        return MAX_IT, False

    def _base_solve(self):
        dss.Command("Set ControlMode=STATIC")
        dss.Solution.Solve()
        if getattr(self, "inv", None):
            self._iterate(fault=False)

    # ------------------------------------------------------------------ fault sample
    def _apply_fault(self, bus, ftype, phs, rf, currents=True):
        if ftype == "SLG":
            cmd = f"Edit Fault.F1 phases=1 bus1={bus}.{phs[0]} bus2={bus}.0 r={rf}"
        elif ftype == "LL":
            cmd = f"Edit Fault.F1 phases=1 bus1={bus}.{phs[0]} bus2={bus}.{phs[1]} r={rf}"
        elif ftype == "LLG":
            cmd = f"Edit Fault.F1 phases=2 bus1={bus}.{phs[0]}.{phs[1]} bus2={bus}.0.0 r={rf}"
        else:
            cmd = f"Edit Fault.F1 phases=3 bus1={bus}.1.2.3 bus2={bus}.0.0.0 r={rf}"
        dss.Command(cmd + " enabled=yes")
        dss.Command("Set ControlMode=OFF")
        dss.Solution.Solve()
        it, ok = self._iterate(fault=True)
        converged = dss.Solution.Converged() and ok
        self.iters.append(it); self.ok.append(ok)
        self.inv_state = [(seq(inv["cur"] / inv["ir"])[1], seq(inv["cur"] / inv["ir"])[2]) for inv in self.inv]
        Vf, If = self._snapshot(currents)
        dss.Command("Fault.F1.enabled=no")
        self._base_solve()
        return Vf, If, converged


def generate(n_bus, n_samples, seed, out_path, der_penetration, strategy, imax=1.2, rf_range=(1e-3, 100.0)):
    import os
    out_path = os.path.abspath(out_path)
    f = InvFeeder(n_bus, der_penetration, strategy=strategy, imax=imax, seed=seed)
    rng = np.random.RandomState(seed)
    recs, inv1, inv2 = [], [], []
    for _ in range(n_samples):
        r = f.sample(rng, rf_range=rf_range)
        if r["converged"]:
            recs.append(r)
            inv1.append([abs(a) for a, _ in f.inv_state]); inv2.append([abs(b) for _, b in f.inv_state])
    np.savez_compressed(
        out_path,
        Vpre=np.stack([r["Vpre"] for r in recs]), Vf=np.stack([r["Vf"] for r in recs]),
        Ipre=np.stack([r["Ipre"] for r in recs]), If=np.stack([r["If"] for r in recs]),
        ftype=np.array([r["ftype"] for r in recs]), bus=np.array([r["bus"] for r in recs]),
        phases=np.array([",".join(map(str, r["phases"])) for r in recs]),
        rf=np.array([r["rf"] for r in recs]), scale=np.array([r["scale"] for r in recs]),
        node_names=np.array(list(f.node_index)), buses=np.array(f.buses),
        lines=np.array(f.lines), line_buses=np.array(f.line_buses),
        substation=simlib.SUBSTATION[n_bus], der_buses=np.array(f.der_buses, dtype=str),
        der_penetration=der_penetration, seed=seed, strategy=strategy, imax=imax,
        inv_I1=np.array(inv1), inv_I2=np.array(inv2),
        iters=np.array(f.iters), iter_ok=np.array(f.ok),
    )
    return len(recs), float(np.mean(f.ok)), float(np.mean(f.iters))
"""

if RUN.get("H") and not HAVE_DSS:
    print("SKIP part H: OpenDSS is not installed. Attach the dataset 'opendss-wheels' or turn Internet ON.")
if RUN.get("H") and HAVE_DSS:
    get_simlib()                                   # writable feeder copies, case-safe names
    with open(os.path.join(WORK, "invlib.py"), "w") as fh:
        fh.write(INVLIB)
    sys.path.insert(0, WORK)
    import importlib, invlib; importlib.reload(invlib)
    rows, stats = [], []
    for nb in FEEDERS:
        base = small(K.dataset(nb)); buses = dmp_sets(nb)["DMP"]; obs0 = K.rows_for_buses(base, buses)
        data = {}
        for st in H_STRATEGIES:
            path = os.path.join(SIMDIR, f"ieee{nb}_inv_{st}.npz")
            if not os.path.isfile(path):
                t = time.time()
                n, okr, it = invlib.generate(nb, 600 if SMOKE else N_SIM, 2026, path, H_PEN, st, imax=H_IMAX)
                print(f"simulated ieee{nb} {st}: {n} samples, converged {100*okr:.1f}%, mean iterations {it:.1f}, "
                      f"{(time.time()-t)/60:.1f} min", flush=True)
            z = np.load(path)
            stats.append(dict(feeder=nb, strategy=st, n=int(len(z["rf"])), iter_ok=float(z["iter_ok"].mean()),
                              iters=float(z["iters"].mean()), I1_median=float(np.median(z["inv_I1"])),
                              I2_median=float(np.median(z["inv_I2"])), I2_max=float(z["inv_I2"].max())))
            data[st] = F.Dataset(path)
        for noise in ("1% TVE", "uPMU"):
            for rep in ("SEQ", "FPS"):
                X0 = K.features_of(base, obs0, 0, noise, rep)
                m0 = LGBMClassifier(**K.FAST).fit(X0, base.y)
                dl = data[H_STRATEGIES[0]]; ol = K.rows_for_buses(dl, buses)
                ml = LGBMClassifier(**K.FAST).fit(K.features_of(dl, ol, 0, noise, rep), dl.y)
                for st, ds in data.items():
                    obs = K.rows_for_buses(ds, buses)
                    X = K.features_of(ds, obs, 0, noise, rep)
                    r = K.folds_acc(X, ds.y, ds.rf, seeds=SEEDS)
                    ok0 = m0.predict(X) == ds.y
                    # the "limit" model was trained on this very data set: use its 5-fold result instead
                    okl = None if st == H_STRATEGIES[0] else (ml.predict(X) == ds.y)
                    rows.append(dict(feeder=nb, strategy=st, noise=noise, rep=rep, same_acc=r["acc"], same_sd=r["acc_sd"],
                                     same_hif=r["hif"], noDER_to_this=float(ok0.mean()),
                                     noDER_to_this_hif=float(ok0[ds.rf > 10].mean()),
                                     limit_to_this=r["acc"] if okl is None else float(okl.mean()),
                                     limit_to_this_hif=r["hif"] if okl is None else float(okl[ds.rf > 10].mean())))
                    print(nb, st, noise, rep, f"same {100*r['acc']:.1f} | noDER->this {100*ok0.mean():.1f} | "
                          f"limit->this {100*rows[-1]['limit_to_this']:.1f}", flush=True)
    print(pd.DataFrame(stats).round(3).to_string())
    save("R3_H_inverter_gridcode", dict(results=rows, stats=stats, pen=H_PEN, imax=H_IMAX))

## I. Expected calibration error (comment C6: ECE is listed in Fig. 2 but not reported)
ECE = sum over 15 confidence bins of (share of snapshots in the bin) x |accuracy - mean confidence|,
with confidence = largest predicted class probability (same definition as e15).
DMP three-sensor sets, LightGBM, 2 seeds x 5 folds, AGG / SEQ / FPS, 1% TVE and micro-PMU.
Also reported: accuracy of the 10% least confident snapshots, which links ECE to the reject option.

In [ ]:
def ece(proba, ok, bins=15):
    conf = proba.max(1); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            e += s.mean() * abs(ok[s].mean() - conf[s].mean())
    return float(e)

if RUN.get("I"):
    rows = []
    for nb in FEEDERS:
        ds = small(K.dataset(nb)); obs = K.rows_for_buses(ds, dmp_sets(nb)["DMP"])
        for noise in ("1% TVE", "uPMU"):
            for rep in ("AGG", "SEQ", "FPS"):
                X = K.features_of(ds, obs, 0, noise, rep)
                eces, accs, low10 = [], [], []
                for s_ in SEEDS:
                    for tr, te in StratifiedKFold(5, shuffle=True, random_state=s_).split(X, ds.y):
                        m = LGBMClassifier(**K.FAST).fit(X[tr], ds.y[tr])
                        pr = m.predict_proba(X[te]); ok = pr.argmax(1) == ds.y[te]
                        eces.append(ece(pr, ok)); accs.append(ok.mean())
                        cut = np.quantile(pr.max(1), 0.10); low10.append(ok[pr.max(1) <= cut].mean())
                rows.append(dict(feeder=nb, noise=noise, rep=rep, ece=float(np.mean(eces)), ece_sd=float(np.std(eces)),
                                 acc=float(np.mean(accs)), acc_low10=float(np.mean(low10))))
                print(nb, noise, rep, f"ECE {100*np.mean(eces):.2f} +- {100*np.std(eces):.2f} %  acc {100*np.mean(accs):.1f}"
                      f"  acc of 10% least confident {100*np.mean(low10):.1f}", flush=True)
    save("R3_I_ece", rows)

In [ ]:
shutil.make_archive(os.path.join(WORK, "R3_results"), "zip", RESULTS)
print("wrote /kaggle/working/R3_results.zip   total", round((time.time() - t0) / 3600, 2), "h")